# 🎙️ DeepBrief.AI — Stage 4: Sentiment & Emotion Analysis

Analyses the emotional tone of each meeting — both **per segment** and as an **overall meeting summary**.

**Key improvement over naive per-clip analysis:**  
Short AMI clips (2–5 words each) carry too little context for reliable sentiment/emotion scoring.  
Stage 4 first **groups all clips by meeting ID** and **concatenates their transcripts** into one  
full meeting text before running the models. This produces significantly more accurate and  
meaningful tone scores.

**Two models:**
- `cardiffnlp/twitter-roberta-base-sentiment-latest` — Sentiment (Positive / Neutral / Negative)
- `j-hartmann/emotion-english-distilroberta-base` — Emotion (anger, disgust, fear, joy, neutral, sadness, surprise)

| | |
|---|---|
| **Input** | `data/transcripts.json` (Stage 2) + `data/stage3_output.json` (Stage 3) |
| **Output** | `data/stage4_output.json` + `data/stage4_summary.csv` |
| **Analysis unit** | Full meeting (all clips concatenated) + per-segment breakdown |

**Run Stages 1, 2, and 3 first.**

## ⚙️ Imports & Configuration

In [1]:
import json
import re
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from transformers import pipeline

warnings.filterwarnings("ignore")

# ─── Paths ────────────────────────────────────────────────────────────────────
TRANSCRIPTS_PATH = Path("data/transcripts.json")     # Stage 2 output
STAGE3_PATH      = Path("data/stage3_output.json")   # Stage 3 output
OUTPUT_PATH      = Path("data/stage4_output.json")   # Stage 4 full output
CSV_PATH         = Path("data/stage4_summary.csv")   # Stage 4 summary CSV

# ─── Models ───────────────────────────────────────────────────────────────────
SENTIMENT_MODEL  = "cardiffnlp/twitter-roberta-base-sentiment-latest"
EMOTION_MODEL    = "j-hartmann/emotion-english-distilroberta-base"

# ─── Chunking — both models have 512 token limit ─────────────────────────────
MAX_CHUNK_WORDS  = 100

# ─── Device ───────────────────────────────────────────────────────────────────
DEVICE      = 0 if torch.cuda.is_available() else -1
device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"

print("=" * 60)
print("DeepBrief.AI — Stage 4: Sentiment & Emotion Analysis")
print("=" * 60)
print(f"  Device          : {device_name}")
print(f"  Sentiment model : {SENTIMENT_MODEL}")
print(f"  Emotion model   : {EMOTION_MODEL}")
print(f"  Input (Stage 2) : {TRANSCRIPTS_PATH}")
print(f"  Input (Stage 3) : {STAGE3_PATH}")
print(f"  Output          : {OUTPUT_PATH}")

c:\Users\sidsu\Desktop\Sem 4\AI\AT3\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DeepBrief.AI — Stage 4: Sentiment & Emotion Analysis
  Device          : NVIDIA GeForce RTX 4060 Laptop GPU
  Sentiment model : cardiffnlp/twitter-roberta-base-sentiment-latest
  Emotion model   : j-hartmann/emotion-english-distilroberta-base
  Input (Stage 2) : data\transcripts.json
  Input (Stage 3) : data\stage3_output.json
  Output          : data\stage4_output.json


## ✅ Validate & Load Inputs

In [2]:
# ── Validate ──────────────────────────────────────────────────────────────────
issues = []
if not TRANSCRIPTS_PATH.exists():
    issues.append(f"Stage 2 output not found: {TRANSCRIPTS_PATH}")
if not STAGE3_PATH.exists():
    issues.append(f"Stage 3 output not found: {STAGE3_PATH}")
if issues:
    for i in issues: print(f"  [✗] {i}")
    raise FileNotFoundError("Missing required input files. See above.")

# ── Load Stage 2 transcripts ──────────────────────────────────────────────────
with open(TRANSCRIPTS_PATH, "r", encoding="utf-8") as f:
    transcripts = json.load(f)

# Normalise id field — Stage 2 uses sample_id, Stage 3 uses id
for entry in transcripts:
    if "id" not in entry and "sample_id" in entry:
        entry["id"] = entry["sample_id"]

# ── Load Stage 3 insights ─────────────────────────────────────────────────────
with open(STAGE3_PATH, "r", encoding="utf-8") as f:
    stage3_data = json.load(f)
stage3_lookup = {r["id"]: r for r in stage3_data}

print(f"  [✓] Stage 2 loaded — {len(transcripts)} clip(s)")
print(f"  [✓] Stage 3 loaded — {len(stage3_data)} record(s)")
print()

# ── Preview ───────────────────────────────────────────────────────────────────
print("First 5 clips:")
for t in transcripts[:5]:
    txt = t.get("transcript", "")
    print(f"  {t['id']:25s} | {len(txt.split()):3d} words | {txt[:60]}")

  [✓] Stage 2 loaded — 200 clip(s)
  [✓] Stage 3 loaded — 10 record(s)

First 5 clips:
  EN2001a_clip00            |  18 words | if you SS agent they have this big warning about doing nothi
  EN2001a_clip01            |   2 words | Hardly any.
  EN2001a_clip02            |  13 words | because the wave data are obviously not going to get off the
  EN2001a_clip03            |  20 words | Yeah, it'll play them in some order in which they were sad, 
  EN2001a_clip04            |   4 words | all these fancy pants


## 🔗 Group Clips by Meeting ID

This is the key fix. Short individual clips (2–5 words) don't carry enough  
context for reliable sentiment/emotion scoring. Grouping all clips from the  
same meeting into one full transcript gives the models much more to work with.

**Example:**
```
EN2001a_clip00: "Yeah exactly."            (2 words)
EN2001a_clip01: "So what do we do next?"   (7 words)
                        ↓  concatenated
EN2001a: "Yeah exactly. So what do we do next?..." (9+ words, growing)
```

In [3]:
def get_meeting_id(clip_id: str) -> str:
    """
    Extracts the meeting ID from a clip ID.
    e.g. 'EN2001a_clip00' -> 'EN2001a'
         'ES2002b_clip03' -> 'ES2002b'
    Falls back to the full clip ID if no pattern matches.
    """
    match = re.match(r'^([A-Z]{2}\d{4}[a-z])_clip\d+$', clip_id)
    if not match:
        print(f"  [WARN] Could not parse meeting ID from: {clip_id} — treating as standalone")
    return match.group(1) if match else clip_id


In [4]:
# ── Group all clips by meeting ID ─────────────────────────────────────────────
meeting_groups = defaultdict(lambda: {
    "clips":       [],   # list of clip records
    "clip_ids":    [],   # ordered clip IDs
    "transcripts": [],   # ordered transcript texts
})

In [5]:
for clip in transcripts:
    mid = get_meeting_id(clip["id"])
    meeting_groups[mid]["clips"].append(clip)
    meeting_groups[mid]["clip_ids"].append(clip["id"])
    txt = clip.get("transcript", "").strip()
    if txt:
        meeting_groups[mid]["transcripts"].append(txt)

print(f"  [✓] {len(transcripts)} clips grouped into {len(meeting_groups)} meetings")
print()
print("Meeting breakdown:")
print(f"  {'Meeting ID':<15} {'Clips':>6} {'Total words':>12} {'Combined preview'}")
print(f"  {'-'*15} {'-'*6} {'-'*12} {'-'*40}")
for mid, data in meeting_groups.items():
    combined = " ".join(data["transcripts"])
    print(f"  {mid:<15} {len(data['clips']):>6} {len(combined.split()):>12} {combined[:50]}...")

  [✓] 200 clips grouped into 10 meetings

Meeting breakdown:
  Meeting ID       Clips  Total words Combined preview
  --------------- ------ ------------ ----------------------------------------
  EN2001a             20          266 if you SS agent they have this big warning about d...
  EN2001b             20          255 This is how much time you spend just getting the r...
  EN2001d             20          237 It's ready to get started. And this is summarized ...
  EN2001e             20          252 but it's probably not a simple thing. The interest...
  EN2003a             20          219 Because, I mean, would six hours a day be pushing ...
  EN2004a             20          191 So would you have been tempted to put an asterisk ...
  EN2005a             20          243 happening in the meeting. but that was not well pi...
  EN2006a             20          169 But it's actually Merlin and all the international...
  EN2006b             20          141 Yes, assuming it's talking abou

## 🤖Load Models

In [6]:
print("Loading sentiment model (first run ~250MB)...")
sentiment_pipe = pipeline(
    "sentiment-analysis",
    model=SENTIMENT_MODEL,
    device=DEVICE,
    top_k=None,
    truncation=True,
    max_length=512,
    model_kwargs={"use_safetensors": True},
)
print(f"  [✓] {SENTIMENT_MODEL}")

Loading sentiment model (first run ~250MB)...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2436.51it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [✓] cardiffnlp/twitter-roberta-base-sentiment-latest


In [7]:
print("Loading emotion model (first run ~250MB)...")
emotion_pipe = pipeline(
    "text-classification",
    model=EMOTION_MODEL,
    device=DEVICE,
    top_k=None,
    truncation=True,
    max_length=512,
    model_kwargs={"use_safetensors": True},
)
print(f"  [✓] {EMOTION_MODEL}")

Loading emotion model (first run ~250MB)...


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1997.81it/s]


  [✓] j-hartmann/emotion-english-distilroberta-base


## 🔧 Helper Functions

In [8]:
EMOTION_LABELS = ["anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"]

In [9]:
def chunk_text(text: str, max_words: int = MAX_CHUNK_WORDS) -> list:
    """Splits long text into chunks at sentence boundaries."""
    if len(text.split()) <= max_words:
        return [text]
    sentences = re.split(r'(?<=[.!?])\s+', text)
    chunks, cur, count = [], [], 0
    for s in sentences:
        wc = len(s.split())
        if count + wc > max_words and cur:
            chunks.append(" ".join(cur))
            cur, count = [s], wc
        else:
            cur.append(s)
            count += wc
    if cur:
        chunks.append(" ".join(cur))
    return chunks or [text]

In [10]:
def score_sentiment(text: str) -> dict:
    """Sentiment analysis with chunking. Returns {positive, neutral, negative, label}.

    Each chunk returns 3 softmax scores that sum to 1.0. Scores are collected
    per-chunk as a unit and averaged, keeping the distribution valid.
    """
    if not text or not text.strip():
        return {"positive": 0.0, "neutral": 1.0, "negative": 0.0, "label": "neutral"}

    chunk_scores = []
    for chunk in chunk_text(text):
        try:
            raw = {item["label"].lower(): item["score"] for item in sentiment_pipe(chunk)[0]}
            # Normalise label variants (pos/neu/neg vs positive/neutral/negative)
            chunk_scores.append({
                "positive": raw.get("positive", raw.get("pos", 0.0)),
                "neutral":  raw.get("neutral",  raw.get("neu", 0.0)),
                "negative": raw.get("negative", raw.get("neg", 0.0)),
            })
        except Exception:
            continue

    if not chunk_scores:
        return {"positive": 0.0, "neutral": 1.0, "negative": 0.0, "label": "neutral"}

    scores = {
        k: round(float(np.mean([c[k] for c in chunk_scores])), 4)
        for k in ["positive", "neutral", "negative"]
    }
    scores["label"] = max(["positive", "neutral", "negative"], key=lambda k: scores[k])
    return scores


In [11]:
def score_emotion(text: str) -> dict:
    """Emotion classification with chunking. Returns 7 emotion scores + dominant."""
    if not text or not text.strip():
        return {**{l: 0.0 for l in EMOTION_LABELS}, "neutral": 1.0, "dominant": "neutral"}
    all_scores = defaultdict(list)
    for chunk in chunk_text(text):
        try:
            for item in emotion_pipe(chunk)[0]:
                all_scores[item["label"].lower()].append(item["score"])
        except Exception:
            continue
    scores = {
        l: round(float(np.mean(all_scores[l])) if all_scores[l] else 0.0, 4)
        for l in EMOTION_LABELS
    }
    scores["dominant"] = max(EMOTION_LABELS, key=lambda k: scores[k])
    return scores

In [12]:
def aggregate_scores(score_list: list, score_type: str) -> dict:
    """Averages a list of score dicts into one overall dict."""
    if not score_list:
        return {}
    if score_type == "sentiment":
        keys = ["positive", "neutral", "negative"]
        agg  = {k: round(float(np.mean([s[k] for s in score_list if k in s])), 4) for k in keys}
        agg["label"] = max(keys, key=lambda k: agg[k])
        return agg
    if score_type == "emotion":
        agg = {l: round(float(np.mean([s[l] for s in score_list if l in s])), 4) for l in EMOTION_LABELS}
        agg["dominant"] = max(EMOTION_LABELS, key=lambda k: agg[k])
        return agg
    return {}

In [13]:
print("Helper functions defined")
# Sanity check
test = "The project is going really well and we are all very happy with the progress."
print(f"  Sanity check sentiment : {score_sentiment(test)}")
print(f"  Sanity check emotion   : {score_emotion(test)}")

Helper functions defined
  Sanity check sentiment : {'positive': 0.9885, 'neutral': 0.0088, 'negative': 0.0027, 'label': 'positive'}
  Sanity check emotion   : {'anger': 0.0015, 'disgust': 0.001, 'fear': 0.0002, 'joy': 0.9865, 'neutral': 0.0064, 'sadness': 0.0017, 'surprise': 0.0027, 'dominant': 'joy'}


## 🔍 Cell 7 — Segment Extractor

Splits the full concatenated meeting transcript into sentence-level segments  
for per-segment tone scoring.

In [14]:
def extract_segments(full_text: str) -> list:
    """
    Splits a full meeting transcript into sentence-level segments.
    Filters out segments shorter than 4 words.
    Falls back to whole-text single segment if splitting yields nothing.
    """
    if not full_text or not full_text.strip():
        return []

    sentences = [
        s.strip() for s in re.split(r'(?<=[.!?])\s+', full_text.strip())
        if s.strip() and len(s.split()) >= 4
    ]

    if not sentences:
        return [{"text": full_text.strip(), "segment_index": 0,
                 "word_count": len(full_text.split()), "speaker": "FULL_MEETING"}]

    return [
        {"text": s, "segment_index": i,
         "word_count": len(s.split()), "speaker": f"SEG_{i:02d}"}
        for i, s in enumerate(sentences)
    ]


print("  [✓] Segment extractor defined")

  [✓] Segment extractor defined


## 🔬 Cell 8 — Run Full Analysis

For each meeting:
1. Concatenates all clip transcripts into one full meeting text
2. Runs per-segment sentiment + emotion scoring
3. Aggregates to overall meeting-level scores
4. Merges with Stage 3 insights
5. Also stores individual clip-level results for reference

In [15]:
results = []

print(f"Analysing {len(meeting_groups)} meeting(s)...")
print("=" * 60)

for meeting_id, data in tqdm(meeting_groups.items(), desc="Meetings"):

    clips      = data["clips"]
    clip_ids   = data["clip_ids"]

    # ── Concatenate all clip transcripts into one full meeting text ────────────
    full_transcript = " ".join(data["transcripts"]).strip()

    if not full_transcript:
        print(f"  [SKIP] {meeting_id} — all clips empty")
        continue

    total_words = len(full_transcript.split())
    print(f"  {meeting_id} — {len(clips)} clip(s) → {total_words} words combined")

    # ── Pull Stage 3 insights ─────────────────────────────────────────────────
    # Stage 3 now outputs one record per MEETING (not per clip).
    # Look up by meeting_id directly instead of iterating over clip_ids.
    s3 = stage3_lookup.get(meeting_id, {})
    summary_data = s3.get("summary", {})
    actions_data = s3.get("actions", {})

    merged_summary = {
        "summary":    summary_data.get("summary",    ""),
        "key_points": summary_data.get("key_points", []),
        "decisions":  summary_data.get("decisions",  []),
    }
    merged_actions = {"action_items": actions_data.get("action_items", [])}

    # ── Extract sentence segments from full meeting text ──────────────────────
    segments = extract_segments(full_transcript)

    # ── Score every segment ───────────────────────────────────────────────────
    per_segment = []
    for seg in segments:
        per_segment.append({
            "segment_index": seg["segment_index"],
            "speaker":       seg["speaker"],
            "text":          seg["text"],
            "word_count":    seg["word_count"],
            "sentiment":     score_sentiment(seg["text"]),
            "emotion":       score_emotion(seg["text"]),
        })

    # ── Score full meeting text directly (most accurate overall score) ────────
    meeting_sentiment = score_sentiment(full_transcript)
    meeting_emotion   = score_emotion(full_transcript)

    overall = {
        "sentiment":       meeting_sentiment,
        "emotion":         meeting_emotion,
        "total_segments":  len(per_segment),
        "total_words":     total_words,
        "total_clips":     len(clips),
    }

    # ── Per-clip results for reference ────────────────────────────────────────
    per_clip = []
    for clip in clips:
        clip_txt = clip.get("transcript", "").strip()
        per_clip.append({
            "clip_id":    clip["id"],
            "text":       clip_txt,
            "word_count": len(clip_txt.split()),
            "sentiment":  score_sentiment(clip_txt) if clip_txt else {},
            "emotion":    score_emotion(clip_txt)   if clip_txt else {},
        })

    # ── Assemble merged record ────────────────────────────────────────────────
    results.append({
        "id":               meeting_id,
        "clip_ids":         clip_ids,
        "meeting_id":       clips[0].get("meeting_id", meeting_id),
        "full_transcript":  full_transcript,
        "summary":          merged_summary,
        "actions":          merged_actions,
        "per_segment":      per_segment,
        "per_clip":         per_clip,
        "overall":          overall,
    })

print()
print("=" * 60)
print(f"Analysis complete — {len(results)} meeting(s) processed")

Analysing 10 meeting(s)...


Meetings:   0%|          | 0/10 [00:00<?, ?it/s]

  EN2001a — 20 clip(s) → 266 words combined


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Meetings:  10%|█         | 1/10 [00:04<00:36,  4.07s/it]

  EN2001b — 20 clip(s) → 255 words combined


...

  EN2006b — 20 clip(s) → 141 words combined


Meetings:  90%|█████████ | 9/10 [00:21<00:02,  2.00s/it]

  EN2009b — 20 clip(s) → 177 words combined


Meetings: 100%|██████████| 10/10 [00:23<00:00,  2.39s/it]


Analysis complete — 10 meeting(s) processed


## 💾 Cell 9 — Save Output

In [16]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"  [✓] Output saved → {OUTPUT_PATH}")
print(f"  [✓] Records written: {len(results)}")

  [✓] Output saved → data\stage4_output.json
  [✓] Records written: 10


## 🖨️ Cell 10 — Validation & Preview

In [17]:
with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
    verification = json.load(f)

print(f"Verification: {len(verification)} meeting record(s) readable from disk.")

if verification:
    rec     = verification[0]
    overall = rec.get("overall", {})
    sent    = overall.get("sentiment", {})
    emot    = overall.get("emotion",   {})
    summary = rec.get("summary", {})
    actions = rec.get("actions", {})

    print()
    print("=" * 60)
    print(f"PREVIEW — Meeting: {rec['id']}")
    print("=" * 60)

    print(f"\n📋 METADATA")
    print(f"   Clips     : {rec.get('clip_ids')}")
    print(f"   Total clips: {overall.get('total_clips')}")
    print(f"   Total words: {overall.get('total_words')} (vs ~3 words per clip before fix!)")
    print(f"   Segments   : {overall.get('total_segments')}")

    print(f"\n🗣️  FULL TRANSCRIPT (first 200 chars)")
    print(f"   {rec.get('full_transcript','')[:200]}...")

    print(f"\n📝 STAGE 3 — MERGED INSIGHTS")
    print(f"   Summary      : {summary.get('summary','')[:120]}...")
    print(f"   Key points   : {len(summary.get('key_points',[]))}")
    print(f"   Decisions    : {len(summary.get('decisions',[]))}")
    print(f"   Action items : {len(actions.get('action_items',[]))}")

    print(f"\n😊 STAGE 4 — OVERALL MEETING TONE (full transcript analysis)")
    print(f"   Sentiment : {sent.get('label','N/A').upper()} "
          f"(pos={sent.get('positive',0):.3f}, "
          f"neu={sent.get('neutral',0):.3f}, "
          f"neg={sent.get('negative',0):.3f})")
    print(f"   Emotion   : {emot.get('dominant','N/A').upper()} "
          f"(joy={emot.get('joy',0):.3f}, "
          f"anger={emot.get('anger',0):.3f}, "
          f"neutral={emot.get('neutral',0):.3f})")

    print(f"\n📊 FIRST 3 SEGMENTS")
    for seg in rec.get("per_segment", [])[:3]:
        s = seg.get("sentiment", {})
        e = seg.get("emotion",   {})
        print(f"   [{seg['segment_index']}] {seg['word_count']} words — {seg.get('text','')[:80]}")
        print(f"        Sentiment: {s.get('label','N/A').upper()} | Emotion: {e.get('dominant','N/A').upper()}")

    print()
    print("Stage 4 complete. Proceed to Stage 5 (Dashboard).")

Verification: 10 meeting record(s) readable from disk.

PREVIEW — Meeting: EN2001a

📋 METADATA
   Clips     : ['EN2001a_clip00', 'EN2001a_clip01', 'EN2001a_clip02', 'EN2001a_clip03', 'EN2001a_clip04', 'EN2001a_clip05', 'EN2001a_clip06', 'EN2001a_clip07', 'EN2001a_clip08', 'EN2001a_clip09', 'EN2001a_clip10', 'EN2001a_clip11', 'EN2001a_clip12', 'EN2001a_clip13', 'EN2001a_clip14', 'EN2001a_clip15', 'EN2001a_clip16', 'EN2001a_clip17', 'EN2001a_clip18', 'EN2001a_clip19']
   Total clips: 20
   Total words: 266 (vs ~3 words per clip before fix!)
   Segments   : 18

🗣️  FULL TRANSCRIPT (first 200 chars)
   if you SS agent they have this big warning about doing nothing at all in the gateway machine. Hardly any. because the wave data are obviously not going to get off there completely. Yeah, it'll play th...

📝 STAGE 3 — MERGED INSIGHTS
   Summary      : The participants discussed technical considerations for representing series data, including warnings about gateway machi...
   Key points   : 1

## 📊 Cell 11 — Corpus Summary & CSV Export

In [18]:
rows = []
for rec in verification:
    overall = rec.get("overall", {})
    sent    = overall.get("sentiment", {})
    emot    = overall.get("emotion",   {})
    summary = rec.get("summary", {})
    actions = rec.get("actions", {})

    rows.append({
        "meeting_id":       rec["id"],
        "total_clips":      overall.get("total_clips",    0),
        "total_words":      overall.get("total_words",    0),
        "total_segments":   overall.get("total_segments", 0),
        "n_key_points":     len(summary.get("key_points",  [])),
        "n_decisions":      len(summary.get("decisions",   [])),
        "n_action_items":   len(actions.get("action_items",[])),
        "sentiment_label":  sent.get("label",    "N/A"),
        "sent_positive":    sent.get("positive", 0.0),
        "sent_neutral":     sent.get("neutral",  0.0),
        "sent_negative":    sent.get("negative", 0.0),
        "dominant_emotion": emot.get("dominant", "N/A"),
        "emot_joy":         emot.get("joy",      0.0),
        "emot_anger":       emot.get("anger",    0.0),
        "emot_neutral":     emot.get("neutral",  0.0),
        "emot_sadness":     emot.get("sadness",  0.0),
        "emot_surprise":    emot.get("surprise", 0.0),
        "emot_fear":        emot.get("fear",     0.0),
        "emot_disgust":     emot.get("disgust",  0.0),
    })

df = pd.DataFrame(rows)

print("=" * 60)
print("CORPUS-LEVEL SUMMARY (meeting level)")
print("=" * 60)
print(df[["meeting_id", "total_words", "sentiment_label",
          "dominant_emotion", "sent_positive", "sent_negative",
          "emot_joy", "emot_anger"]].to_string(index=False))

print("\nSENTIMENT DISTRIBUTION:")
print(df["sentiment_label"].value_counts().to_string())

print("\nDOMINANT EMOTION DISTRIBUTION:")
print(df["dominant_emotion"].value_counts().to_string())

print("\nMEAN SCORES ACROSS ALL MEETINGS:")
num_cols = ["sent_positive","sent_neutral","sent_negative",
            "emot_joy","emot_anger","emot_neutral",
            "emot_sadness","emot_surprise","emot_fear","emot_disgust"]
print(df[num_cols].mean().round(4).to_string())

print("\nSTAGE 3 COVERAGE:")
print(f"   Total action items : {df['n_action_items'].sum()}")
print(f"   Total decisions    : {df['n_decisions'].sum()}")
print(f"   Total key points   : {df['n_key_points'].sum()}")

df.to_csv(CSV_PATH, index=False)
print(f"\n  [✓] Summary CSV saved → {CSV_PATH}")
print()
print("✅ Stage 4 complete. Proceed to Stage 5.")

CORPUS-LEVEL SUMMARY (meeting level)
meeting_id  total_words sentiment_label dominant_emotion  sent_positive  sent_negative  emot_joy  emot_anger
   EN2001a          266         neutral          neutral         0.0479         0.3050    0.0497      0.0094
   EN2001b          255         neutral          neutral         0.1087         0.1205    0.0027      0.0134
   EN2001d          237         neutral          neutral         0.3393         0.0463    0.0176      0.0388
   EN2001e          252         neutral          neutral         0.0835         0.2131    0.0204      0.0102
   EN2003a          219         neutral          neutral         0.1446         0.1104    0.0052      0.0135
   EN2004a          191         neutral          neutral         0.1285         0.2114    0.0037      0.0356
   EN2005a          243        negative          neutral         0.0280         0.4891    0.0028      0.0216
   EN2006a          169         neutral          neutral         0.0935         0.2131    0